# ForeSight-EK — Colab runner

Sets up a Colab session for training: GPU check, clone + install the repo, mount Drive, log in to W&B, run the tests.

**Run the cells top to bottom.** Runtime > Change runtime type > GPU (T4 / L4) first.

## 0. Settings — edit these once

In [ ]:
GITHUB_USER = "AhmadTawil1"  # your GitHub username
REPO_NAME = "foresight-ek"
BRANCH = "main"

WANDB_PROJECT = "foresight-ek"
DRIVE_ROOT = "/content/drive/MyDrive/foresight-ek"  # cache, checkpoints
REPO_DIR = f"/content/{REPO_NAME}"
REPO_URL = f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git"
print(REPO_URL)

## 1. GPU check

In [ ]:
!nvidia-smi

import sys

import torch

print("python:", sys.version.split()[0])
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))
    print("bf16 supported:", torch.cuda.is_bf16_supported())
else:
    print("NO GPU -> Runtime > Change runtime type > GPU")

## 2. Clone (or update) the repo and install it

In [ ]:
import os
import subprocess


def run(cmd, cwd=None):
    print("$", " ".join(cmd))
    subprocess.run(cmd, cwd=cwd, check=True)


if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    run(["git", "fetch", "origin"], cwd=REPO_DIR)
    run(["git", "checkout", BRANCH], cwd=REPO_DIR)
    run(["git", "pull", "--ff-only", "origin", BRANCH], cwd=REPO_DIR)
else:
    run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR])

run(["git", "log", "--oneline", "-1"], cwd=REPO_DIR)
os.chdir(REPO_DIR)
print("cwd:", os.getcwd())

In [ ]:
# Editable install. Colab already ships torch, so we do not reinstall it.
!pip install -q -e . wandb pytest

import anticip

print("anticip imported from:", anticip.__file__)

## 3. Mount Drive

Drive holds the window cache and the checkpoints, so they survive when the Colab session dies.

In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

for sub in ("features", "cache", "checkpoints"):
    Path(DRIVE_ROOT, sub).mkdir(parents=True, exist_ok=True)

!df -h /content/drive/MyDrive | tail -1
!ls -la "$DRIVE_ROOT" 2>/dev/null || ls -la {DRIVE_ROOT}

## 4. Weights & Biases login

Put your key in the Colab sidebar: **key icon (Secrets) > Add new secret**, name `WANDB_API_KEY`, then turn on notebook access.

In [ ]:
import os

import wandb
from google.colab import userdata

try:
    os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
    ok = wandb.login()
except Exception as e:  # noqa: BLE001
    print("secret not found, falling back to interactive login:", e)
    ok = wandb.login()

os.environ["WANDB_PROJECT"] = WANDB_PROJECT
print("wandb logged in:", ok)

In [ ]:
# Smoke test: one tiny run should appear in the W&B project, then be deleted by hand later.
run = wandb.init(project=WANDB_PROJECT, name="colab-smoke", job_type="smoke", tags=["setup"])
wandb.log({"hello": 1.0})
print("run url:", run.url)
run.finish()

## 5. Tests

In [ ]:
!python -m pytest -q

## 6. Session summary

In [ ]:
import sys

import torch

commit = subprocess.run(
    ["git", "rev-parse", "--short", "HEAD"], cwd=REPO_DIR, capture_output=True, text=True
).stdout.strip()
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu only"

print(f"repo      : {REPO_DIR} @ {commit}")
print(f"python    : {sys.version.split()[0]}")
print(f"torch     : {torch.__version__}")
print(f"gpu       : {gpu}")
print(f"drive     : {DRIVE_ROOT}")
print(f"wandb proj: {WANDB_PROJECT}")